In [2]:
from pathlib import Path
import os, json, math, random, shutil
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw, ImageFont
import matplotlib.pyplot as plt

# =========================================================
# SETTINGS
# =========================================================
RANDOM_SEED = 42

# Final 15-image pack:
# 5 baseline + 5 landmark_reg + 4 freq_ablation + 1 personal image = 15
N_BASELINE = 5
N_LANDMARK = 5
N_FREQ = 4
N_PERSONAL = 1

THUMB_SIZE = 256
GRID_COLS = 5

# =========================================================
# PATHS
# =========================================================
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

tables_dir = ROOT / "results" / "tables"
metrics_csv = tables_dir / "metrics_summary.csv"
metrics_json = tables_dir / "metrics_summary.json"

generated_eval_root = ROOT / "results" / "metrics_eval" / "generated_eval"
baseline_eval_dir = generated_eval_root / "baseline"
landmark_eval_dir = generated_eval_root / "landmark_reg"
freq_eval_dir = generated_eval_root / "freq_ablation"

personal_raw_dir = ROOT / "data" / "raw" / "personal_raw"
personal_aligned_dir = ROOT / "data" / "interim" / "personal_aligned"
personal_cropped_dir = ROOT / "data" / "interim" / "personal_cropped"

personal_results_recon = ROOT / "results" / "personal" / "reconstructions"
personal_results_variants = ROOT / "results" / "personal" / "variants"

final_root = ROOT / "results" / "final_submission_pack"
final_root.mkdir(parents=True, exist_ok=True)

final_images_dir = final_root / "final_15_images"
final_images_dir.mkdir(parents=True, exist_ok=True)

figures_dir = ROOT / "figures" / "report"
figures_dir.mkdir(parents=True, exist_ok=True)

manifests_dir = final_root / "manifests"
manifests_dir.mkdir(parents=True, exist_ok=True)

# =========================================================
# HELPERS
# =========================================================
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

def list_images(folder: Path):
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

def choose_deterministic(paths, n, seed=42):
    paths = sorted(paths)
    if len(paths) <= n:
        return paths
    rng = random.Random(seed)
    idxs = sorted(rng.sample(range(len(paths)), n))
    return [paths[i] for i in idxs]

def resize_with_pad(img: Image.Image, size=256, bg="white"):
    img = img.convert("RGB")
    return ImageOps.pad(img, (size, size), color=bg, method=Image.Resampling.LANCZOS)

def center_crop_square(img: Image.Image, crop_ratio=0.92):
    w, h = img.size
    side = int(min(w, h) * crop_ratio)
    side = max(side, 2)
    left = (w - side) // 2
    top = (h - side) // 2
    return img.crop((left, top, left + side, top + side))

def copy_with_new_name(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

def draw_labeled_grid(items, out_path: Path, cols=5, thumb_size=256, title=None):
    """
    items = list of {"image_path": Path, "label": str}
    """
    rows = math.ceil(len(items) / cols)
    label_h = 36
    top_title_h = 40 if title else 0

    canvas = Image.new("RGB", (cols * thumb_size, rows * (thumb_size + label_h) + top_title_h), "white")
    draw = ImageDraw.Draw(canvas)

    try:
        font = ImageFont.load_default()
    except Exception:
        font = None

    if title:
        draw.text((10, 10), title, fill="black", font=font)

    for i, item in enumerate(items):
        r = i // cols
        c = i % cols
        x = c * thumb_size
        y = top_title_h + r * (thumb_size + label_h)

        img = Image.open(item["image_path"]).convert("RGB")
        img = resize_with_pad(img, size=thumb_size)
        canvas.paste(img, (x, y))

        label = item["label"]
        draw.text((x + 6, y + thumb_size + 8), label, fill="black", font=font)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(out_path)
    return out_path

def latest_personal_raw():
    imgs = list_images(personal_raw_dir)
    if len(imgs) == 0:
        return None
    # Prefer me.jpeg if present
    for p in imgs:
        if p.name.lower() == "me.jpeg":
            return p
    return imgs[0]

# =========================================================
# LOAD METRICS + ADD INTERPRETATION
# =========================================================
assert metrics_csv.exists(), f"Missing metrics CSV: {metrics_csv}"
df = pd.read_csv(metrics_csv)

def status_note(row):
    model = row["model"]
    if model == "baseline":
        return "Stable baseline; use as main visual reference."
    if model == "landmark_reg":
        return "Best landmark plausibility; main proposed model; modified FID/KID unstable."
    if model == "freq_ablation":
        return "Optional ablation; geometry improved over baseline; FID/KID unstable."
    return ""

df["status_note"] = df.apply(status_note, axis=1)

metrics_final_csv = final_root / "metrics_summary_for_report.csv"
df.to_csv(metrics_final_csv, index=False)

# =========================================================
# SELECT FINAL 15 IMAGES
# =========================================================
assert baseline_eval_dir.exists(), f"Missing baseline eval dir: {baseline_eval_dir}"
assert landmark_eval_dir.exists(), f"Missing landmark eval dir: {landmark_eval_dir}"
assert freq_eval_dir.exists(), f"Missing freq eval dir: {freq_eval_dir}"

baseline_imgs = choose_deterministic(list_images(baseline_eval_dir), N_BASELINE, seed=RANDOM_SEED + 1)
landmark_imgs = choose_deterministic(list_images(landmark_eval_dir), N_LANDMARK, seed=RANDOM_SEED + 2)
freq_imgs = choose_deterministic(list_images(freq_eval_dir), N_FREQ, seed=RANDOM_SEED + 3)

assert len(baseline_imgs) >= N_BASELINE, "Not enough baseline images"
assert len(landmark_imgs) >= N_LANDMARK, "Not enough landmark_reg images"
assert len(freq_imgs) >= N_FREQ, "Not enough freq_ablation images"

personal_raw = latest_personal_raw()
assert personal_raw is not None, f"No personal image found in {personal_raw_dir}. Put me.jpeg there."

# create processed personal fallbacks if notebook10 outputs are missing
personal_raw_img = Image.open(personal_raw).convert("RGB")
personal_aligned_fallback = final_root / "personal_aligned_fallback.jpg"
personal_cropped_fallback = final_root / "personal_cropped_fallback.jpg"

resize_with_pad(center_crop_square(personal_raw_img, crop_ratio=1.0), THUMB_SIZE).save(personal_aligned_fallback, quality=95)
resize_with_pad(center_crop_square(personal_raw_img, crop_ratio=0.92), THUMB_SIZE).save(personal_cropped_fallback, quality=95)

personal_aligned_candidates = list_images(personal_aligned_dir)
personal_cropped_candidates = list_images(personal_cropped_dir)
personal_recon_candidates = list_images(personal_results_recon)
personal_variant_candidates = list_images(personal_results_variants)

# final 15 selection manifest
selection = []
rank = 1

for i, p in enumerate(baseline_imgs, start=1):
    out_name = f"{rank:02d}_baseline_{i:02d}{p.suffix.lower()}"
    dst = final_images_dir / out_name
    copy_with_new_name(p, dst)
    selection.append({
        "rank": rank,
        "model": "baseline",
        "source_path": str(p),
        "saved_path": str(dst),
        "label": f"baseline_{i:02d}",
        "personal_related": False,
    })
    rank += 1

for i, p in enumerate(landmark_imgs, start=1):
    out_name = f"{rank:02d}_landmark_reg_{i:02d}{p.suffix.lower()}"
    dst = final_images_dir / out_name
    copy_with_new_name(p, dst)
    selection.append({
        "rank": rank,
        "model": "landmark_reg",
        "source_path": str(p),
        "saved_path": str(dst),
        "label": f"landmark_{i:02d}",
        "personal_related": False,
    })
    rank += 1

for i, p in enumerate(freq_imgs, start=1):
    out_name = f"{rank:02d}_freq_ablation_{i:02d}{p.suffix.lower()}"
    dst = final_images_dir / out_name
    copy_with_new_name(p, dst)
    selection.append({
        "rank": rank,
        "model": "freq_ablation",
        "source_path": str(p),
        "saved_path": str(dst),
        "label": f"freq_{i:02d}",
        "personal_related": False,
    })
    rank += 1

# 15th image = personal raw reference
personal_dst = final_images_dir / f"{rank:02d}_personal_me_reference{personal_raw.suffix.lower()}"
copy_with_new_name(personal_raw, personal_dst)
selection.append({
    "rank": rank,
    "model": "personal_reference",
    "source_path": str(personal_raw),
    "saved_path": str(personal_dst),
    "label": "my_face_reference",
    "personal_related": True,
})
rank += 1

assert len(selection) == 15, f"Expected 15 selected items, got {len(selection)}"

selection_df = pd.DataFrame(selection)
selection_csv = manifests_dir / "final_15_manifest.csv"
selection_json = manifests_dir / "final_15_manifest.json"
selection_df.to_csv(selection_csv, index=False)
selection_json.write_text(selection_df.to_json(orient="records", indent=2), encoding="utf-8")

print("Saved final selection manifest:", selection_csv)
print(selection_df[["rank", "model", "label"]])

# =========================================================
# CREATE FINAL 15 GRID
# =========================================================
grid_items = [{"image_path": Path(row["saved_path"]), "label": row["label"]} for _, row in selection_df.iterrows()]
final_grid_path = figures_dir / "final_15_grid.png"
draw_labeled_grid(
    items=grid_items,
    out_path=final_grid_path,
    cols=GRID_COLS,
    thumb_size=THUMB_SIZE,
    title="Final 15-image submission set"
)
print("Saved final 15 grid:", final_grid_path)

# =========================================================
# CREATE QUALITATIVE COMPARISON GRID
# =========================================================
comparison_items = []
for i, p in enumerate(baseline_imgs[:4], start=1):
    comparison_items.append({"image_path": p, "label": f"baseline_{i}"})
for i, p in enumerate(landmark_imgs[:4], start=1):
    comparison_items.append({"image_path": p, "label": f"landmark_{i}"})
for i, p in enumerate(freq_imgs[:4], start=1):
    comparison_items.append({"image_path": p, "label": f"freq_{i}"})

comparison_grid_path = figures_dir / "qualitative_model_comparison_grid.png"
draw_labeled_grid(
    items=comparison_items,
    out_path=comparison_grid_path,
    cols=4,
    thumb_size=THUMB_SIZE,
    title="Qualitative comparison: baseline vs landmark_reg vs freq_ablation"
)
print("Saved qualitative comparison grid:", comparison_grid_path)

# =========================================================
# CREATE PERSONAL PANEL
# =========================================================
personal_panel_items = []

personal_panel_items.append({"image_path": personal_raw, "label": "original_me"})

if len(personal_aligned_candidates) > 0:
    personal_panel_items.append({"image_path": personal_aligned_candidates[0], "label": "aligned_me"})
else:
    personal_panel_items.append({"image_path": personal_aligned_fallback, "label": "aligned_me_fallback"})

if len(personal_cropped_candidates) > 0:
    personal_panel_items.append({"image_path": personal_cropped_candidates[0], "label": "cropped_me"})
else:
    personal_panel_items.append({"image_path": personal_cropped_fallback, "label": "cropped_me_fallback"})

# add inversion/reconstruction if available
if len(personal_recon_candidates) > 0:
    personal_panel_items.append({"image_path": personal_recon_candidates[0], "label": "reconstruction"})

# add up to 3 variants if available
for i, p in enumerate(personal_variant_candidates[:3], start=1):
    personal_panel_items.append({"image_path": p, "label": f"variant_{i}"})

personal_panel_path = figures_dir / "personal_image_panel.png"
draw_labeled_grid(
    items=personal_panel_items,
    out_path=personal_panel_path,
    cols=min(4, len(personal_panel_items)),
    thumb_size=THUMB_SIZE,
    title="Personal image panel"
)
print("Saved personal panel:", personal_panel_path)

# =========================================================
# METRICS TABLE FIGURE
# =========================================================
# Keep only the most relevant columns for the report
report_cols = [
    "model", "fid", "kid_mean", "lpips_mean",
    "landmark_mahalanobis_mean", "status_note"
]
report_df = df[report_cols].copy()

metrics_report_csv = final_root / "metrics_table_for_report.csv"
report_df.to_csv(metrics_report_csv, index=False)

fig, ax = plt.subplots(figsize=(14, 2 + 0.6 * len(report_df)))
ax.axis("off")
tbl = ax.table(
    cellText=report_df.round(4).fillna("unstable").values,
    colLabels=report_df.columns,
    loc="center"
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.5)
metrics_table_png = figures_dir / "metrics_table.png"
plt.tight_layout()
plt.savefig(metrics_table_png, dpi=200, bbox_inches="tight")
plt.close()
print("Saved metrics table figure:", metrics_table_png)

# =========================================================
# WRITE REPORT NOTES
# =========================================================
report_notes = {
    "time": datetime.now().isoformat(),
    "main_model_for_report": "landmark_reg",
    "baseline_model": "baseline",
    "optional_ablation_model": "freq_ablation",
    "interpretation": {
        "baseline": "Stable reference model.",
        "landmark_reg": "Main proposed model; best landmark plausibility among compared variants.",
        "freq_ablation": "Optional ablation; present as exploratory.",
        "fid_kid_note": "Modified-run FID/KID should be described as unstable in the current evaluation setup."
    },
    "personal_note": {
        "raw_image": str(personal_raw),
        "panel_figure": str(personal_panel_path),
        "inversion_outputs_found": len(personal_recon_candidates) + len(personal_variant_candidates)
    },
    "final_assets": {
        "final_15_grid": str(final_grid_path),
        "comparison_grid": str(comparison_grid_path),
        "metrics_table_png": str(metrics_table_png),
        "metrics_report_csv": str(metrics_report_csv),
        "final_selection_manifest_csv": str(selection_csv),
    }
}
report_notes_path = final_root / "report_pack_summary.json"
report_notes_path.write_text(json.dumps(report_notes, indent=2), encoding="utf-8")

# also export a short human-readable note
note_txt = final_root / "report_writing_notes.txt"
note_txt.write_text(
    (
        "Main reporting choice:\n"
        "- Use baseline as stable reference.\n"
        "- Use landmark_reg as the main proposed model.\n"
        "- Use freq_ablation only as optional ablation.\n\n"
        "Metrics interpretation:\n"
        "- Landmark plausibility improved for landmark_reg.\n"
        "- LPIPS diversity did not collapse.\n"
        "- Modified-run FID/KID are unstable and should be described as such.\n\n"
        "Personal image:\n"
        "- me.jpeg has been included in the final pack and personal panel.\n"
        "- If Notebook 10 inversion outputs are added later, rerun this notebook to include them automatically.\n"
    ),
    encoding="utf-8"
)

print("\nSaved report summary:", report_notes_path)
print("Saved writing notes:", note_txt)

# =========================================================
# FINAL PRINT
# =========================================================
print("\nNotebook 11 complete.")
print("Final pack folder:", final_root)
print("Final 15 images folder:", final_images_dir)
print("Main grid:", final_grid_path)
print("Personal panel:", personal_panel_path)
print("Metrics table:", metrics_table_png)
print("Manifest:", selection_csv)

Saved final selection manifest: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/results/final_submission_pack/manifests/final_15_manifest.csv
    rank               model              label
0      1            baseline        baseline_01
1      2            baseline        baseline_02
2      3            baseline        baseline_03
3      4            baseline        baseline_04
4      5            baseline        baseline_05
5      6        landmark_reg        landmark_01
6      7        landmark_reg        landmark_02
7      8        landmark_reg        landmark_03
8      9        landmark_reg        landmark_04
9     10        landmark_reg        landmark_05
10    11       freq_ablation            freq_01
11    12       freq_ablation            freq_02
12    13       freq_ablation            freq_03
13    14       freq_ablation            freq_04
14    15  personal_reference  my_face_reference
Saved final 15 grid: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_gene

In [ ]:
from pathlib import Path
import os, sys, io, json, math, random, shutil, textwrap, subprocess
from datetime import datetime

# =========================================================
# INSTALLS
# =========================================================
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "reportlab", "pandas", "pillow", "matplotlib", "numpy"],
    check=True
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw, ImageFont

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image as RLImage,
    Table, TableStyle, PageBreak
)

# =========================================================
# SETTINGS
# =========================================================
RANDOM_SEED = 42
THUMB = 256
GRID_BG = "white"

# =========================================================
# PATHS
# =========================================================
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

# Main inputs
metrics_csv = ROOT / "results" / "tables" / "metrics_summary.csv"
metrics_json = ROOT / "results" / "tables" / "metrics_summary.json"

generated_eval_root = ROOT / "results" / "metrics_eval" / "generated_eval"
baseline_dir = generated_eval_root / "baseline"
landmark_dir = generated_eval_root / "landmark_reg"
freq_dir = generated_eval_root / "freq_ablation"

personal_raw = ROOT / "data" / "raw" / "personal_raw" / "me.jpeg"
personal_attack_summary = ROOT / "results" / "personal_attack_eval" / "tables" / "personal_attack_summary.json"
personal_attack_pairs = ROOT / "results" / "personal_attack_eval" / "tables" / "personal_attack_pairs.csv"
personal_attack_grid = ROOT / "results" / "personal_attack_eval" / "grids" / "personal_pairs_overview_grid.png"

# Output folder
export_root = ROOT / "results" / "final_exports"
export_root.mkdir(parents=True, exist_ok=True)

fig_root = export_root / "figures_for_pdf"
fig_root.mkdir(parents=True, exist_ok=True)

output_csv = export_root / "final_results_summary.csv"
output_pdf = export_root / "final_results_report.pdf"

# =========================================================
# PRECHECKS
# =========================================================
assert metrics_csv.exists(), f"Missing metrics CSV: {metrics_csv}"
assert baseline_dir.exists(), f"Missing baseline generated_eval dir: {baseline_dir}"
assert landmark_dir.exists(), f"Missing landmark generated_eval dir: {landmark_dir}"
assert freq_dir.exists(), f"Missing freq generated_eval dir: {freq_dir}"

# =========================================================
# HELPERS
# =========================================================
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

def list_images(folder: Path):
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

def choose_n(paths, n, seed=42):
    paths = sorted(paths)
    if len(paths) <= n:
        return paths
    rng = random.Random(seed)
    idxs = sorted(rng.sample(range(len(paths)), n))
    return [paths[i] for i in idxs]

def resize_pad(img: Image.Image, size=256, bg="white"):
    return ImageOps.pad(img.convert("RGB"), (size, size), color=bg, method=Image.Resampling.LANCZOS)

def draw_grid(paths, labels, out_path: Path, cols=4, thumb=256, title=None):
    rows = math.ceil(len(paths) / cols)
    label_h = 28
    title_h = 32 if title else 0

    canvas = Image.new("RGB", (cols * thumb, rows * (thumb + label_h) + title_h), GRID_BG)
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None

    if title:
        draw.text((10, 8), title, fill="black", font=font)

    for i, (p, lab) in enumerate(zip(paths, labels)):
        r = i // cols
        c = i % cols
        x = c * thumb
        y = title_h + r * (thumb + label_h)

        img = Image.open(p).convert("RGB")
        img = resize_pad(img, size=thumb)
        canvas.paste(img, (x, y))
        draw.text((x + 6, y + thumb + 6), lab, fill="black", font=font)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(out_path)
    return out_path

def simple_personal_reference(out_path: Path):
    assert personal_raw.exists(), f"Missing personal image: {personal_raw}"
    img = Image.open(personal_raw).convert("RGB")
    raw_img = resize_pad(img, THUMB)
    crop = resize_pad(ImageOps.fit(img, (THUMB, THUMB), method=Image.Resampling.LANCZOS), THUMB)

    canvas = Image.new("RGB", (THUMB * 2, THUMB + 30), "white")
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None

    canvas.paste(raw_img, (0, 30))
    canvas.paste(crop, (THUMB, 30))
    draw.text((10, 8), "Personal image reference", fill="black", font=font)
    draw.text((10, THUMB + 8), "original_me", fill="black", font=font)
    draw.text((THUMB + 10, THUMB + 8), "processed_preview", fill="black", font=font)
    canvas.save(out_path)
    return out_path

def add_chart(df, ycol, ylabel, title, out_path):
    plt.figure(figsize=(7, 4))
    vals = df[ycol].copy()
    if vals.dtype.kind not in "fi":
        vals = pd.to_numeric(vals, errors="coerce")
    plt.bar(df["model"], vals.fillna(0))
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    return out_path

def rl_image(path, max_width=17*cm, max_height=20*cm):
    img = Image.open(path)
    w, h = img.size
    scale = min(max_width / w, max_height / h)
    return RLImage(str(path), width=w * scale, height=h * scale)

def source_paragraph(text, styles):
    safe = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    return Paragraph(f"<font size=8><b>Source:</b> {safe}</font>", styles["BodySmall"])

# =========================================================
# LOAD + PREPARE SUMMARY DATA
# =========================================================
df = pd.read_csv(metrics_csv)

# Add interpretation columns
def fid_status(x):
    try:
        x = float(x)
        if np.isfinite(x) and x < 1e6:
            return "usable"
        return "unstable"
    except Exception:
        return "unstable"

def kid_status(x):
    try:
        x = float(x)
        if np.isfinite(x):
            return "usable"
        return "unstable"
    except Exception:
        return "unstable"

def interpretation(model):
    if model == "baseline":
        return "Stable baseline; use as main realism reference."
    if model == "landmark_reg":
        return "Best geometry/plausibility; main proposed model."
    if model == "freq_ablation":
        return "Optional ablation; exploratory only."
    return ""

df["fid_status"] = df["fid"].apply(fid_status)
df["kid_status"] = df["kid_mean"].apply(kid_status)
df["interpretation"] = df["model"].apply(interpretation)

# Personal attack summary if available
rows_extra = []
if personal_attack_summary.exists():
    personal_summary = json.loads(personal_attack_summary.read_text(encoding="utf-8"))
    rows_extra.append({
        "model": "personal_attack_eval",
        "snapshot": str(personal_raw) if personal_raw.exists() else "",
        "num_real": np.nan,
        "num_fake": int(personal_summary.get("pair_count", np.nan)),
        "fid": np.nan,
        "kid_mean": np.nan,
        "kid_std": np.nan,
        "lpips_mean": float(personal_summary.get("mean_lpips", np.nan)),
        "lpips_std": np.nan,
        "landmark_success_count": np.nan,
        "landmark_fail_count": np.nan,
        "landmark_mahalanobis_mean": np.nan,
        "landmark_mahalanobis_std": np.nan,
        "fid_status": "",
        "kid_status": "",
        "interpretation": "Own-face perturbation experiment.",
        "attack_success_rate": float(personal_summary.get("attack_success_rate", np.nan)),
        "mean_ssim": float(personal_summary.get("mean_ssim", np.nan)),
        "mean_psnr": float(personal_summary.get("mean_psnr", np.nan)),
    })

if rows_extra:
    df = pd.concat([df, pd.DataFrame(rows_extra)], ignore_index=True)

df.to_csv(output_csv, index=False)

# =========================================================
# CREATE FIGURES FOR PDF
# =========================================================
baseline_samples = choose_n(list_images(baseline_dir), 4, seed=RANDOM_SEED + 1)
landmark_samples = choose_n(list_images(landmark_dir), 4, seed=RANDOM_SEED + 2)
freq_samples = choose_n(list_images(freq_dir), 4, seed=RANDOM_SEED + 3)

baseline_grid = draw_grid(
    baseline_samples,
    [p.stem for p in baseline_samples],
    fig_root / "baseline_grid.png",
    cols=4,
    thumb=THUMB,
    title="Baseline samples"
)

landmark_grid = draw_grid(
    landmark_samples,
    [p.stem for p in landmark_samples],
    fig_root / "landmark_grid.png",
    cols=4,
    thumb=THUMB,
    title="Landmark-regularized samples"
)

freq_grid = draw_grid(
    freq_samples,
    [p.stem for p in freq_samples],
    fig_root / "freq_grid.png",
    cols=4,
    thumb=THUMB,
    title="Frequency ablation samples"
)

comparison_grid = draw_grid(
    baseline_samples[:4] + landmark_samples[:4] + freq_samples[:4],
    [f"b{i+1}" for i in range(len(baseline_samples[:4]))] +
    [f"lm{i+1}" for i in range(len(landmark_samples[:4]))] +
    [f"fq{i+1}" for i in range(len(freq_samples[:4]))],
    fig_root / "qualitative_comparison_grid.png",
    cols=4,
    thumb=THUMB,
    title="Qualitative comparison across models"
)

fid_chart = add_chart(
    df[df["model"].isin(["baseline", "landmark_reg", "freq_ablation"])].copy(),
    "fid",
    "FID (lower is better)",
    "FID comparison",
    fig_root / "fid_chart.png"
)

lpips_chart = add_chart(
    df[df["model"].isin(["baseline", "landmark_reg", "freq_ablation"])].copy(),
    "lpips_mean",
    "LPIPS diversity",
    "LPIPS diversity comparison",
    fig_root / "lpips_chart.png"
)

lm_chart = add_chart(
    df[df["model"].isin(["baseline", "landmark_reg", "freq_ablation"])].copy(),
    "landmark_mahalanobis_mean",
    "Landmark Mahalanobis mean (lower is better)",
    "Landmark plausibility comparison",
    fig_root / "landmark_chart.png"
)

if personal_attack_grid.exists():
    personal_fig = personal_attack_grid
    personal_source = str(personal_attack_grid)
else:
    personal_fig = simple_personal_reference(fig_root / "personal_reference.png")
    personal_source = str(personal_raw)

# =========================================================
# BUILD PDF
# =========================================================
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="TitleCenter", parent=styles["Title"], alignment=TA_CENTER, fontSize=20, spaceAfter=12))
styles.add(ParagraphStyle(name="HeadingSmall", parent=styles["Heading2"], alignment=TA_LEFT, fontSize=13, spaceBefore=8, spaceAfter=6))
styles.add(ParagraphStyle(name="BodySmall", parent=styles["BodyText"], fontSize=9, leading=12))
styles.add(ParagraphStyle(name="BodyCenter", parent=styles["BodyText"], alignment=TA_CENTER, fontSize=10))

doc = SimpleDocTemplate(
    str(output_pdf),
    pagesize=A4,
    rightMargin=1.5*cm,
    leftMargin=1.5*cm,
    topMargin=1.5*cm,
    bottomMargin=1.5*cm,
)

story = []

# Cover
story.append(Paragraph("Final Results Report", styles["TitleCenter"]))
story.append(Paragraph("Realistic Face Image Generation", styles["BodyCenter"]))
story.append(Spacer(1, 0.3*cm))
story.append(Paragraph(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", styles["BodyCenter"]))
story.append(Spacer(1, 0.6*cm))
story.append(Paragraph(
    "This report packages the final experiment outputs into one CSV summary and one PDF, including metrics, model samples, and personal-face figures when available.",
    styles["BodySmall"]
))
story.append(Spacer(1, 0.4*cm))
story.append(Paragraph(f"<b>Main CSV:</b> {output_csv}", styles["BodySmall"]))
story.append(Paragraph(f"<b>Main PDF:</b> {output_pdf}", styles["BodySmall"]))
story.append(Paragraph(f"<b>Metrics source:</b> {metrics_csv}", styles["BodySmall"]))
story.append(PageBreak())

# Metrics summary table
story.append(Paragraph("Metrics Summary", styles["HeadingSmall"]))

table_df = df.copy()
for c in ["fid", "kid_mean", "lpips_mean", "landmark_mahalanobis_mean", "attack_success_rate", "mean_ssim", "mean_psnr"]:
    if c in table_df.columns:
        table_df[c] = pd.to_numeric(table_df[c], errors="coerce").round(4)

cols = [c for c in [
    "model", "fid", "fid_status", "kid_mean", "kid_status",
    "lpips_mean", "landmark_mahalanobis_mean",
    "attack_success_rate", "mean_ssim", "mean_psnr", "interpretation"
] if c in table_df.columns]

table_data = [cols] + table_df[cols].fillna("NA").astype(str).values.tolist()
col_widths = [2.7*cm, 2.1*cm, 2.0*cm, 2.1*cm, 2.0*cm, 2.1*cm, 3.0*cm, 2.3*cm, 2.0*cm, 2.0*cm, 4.2*cm][:len(cols)]

tbl = Table(table_data, colWidths=col_widths, repeatRows=1)
tbl.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D9E8FB")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("FONTSIZE", (0, 0), (-1, -1), 8),
    ("ALIGN", (1, 1), (-2, -1), "CENTER"),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("GRID", (0, 0), (-1, -1), 0.35, colors.grey),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.lightgrey]),
    ("LEFTPADDING", (0,0), (-1,-1), 4),
    ("RIGHTPADDING", (0,0), (-1,-1), 4),
    ("TOPPADDING", (0,0), (-1,-1), 5),
    ("BOTTOMPADDING", (0,0), (-1,-1), 5),
]))
story.append(tbl)
story.append(Spacer(1, 0.25*cm))
story.append(source_paragraph(str(metrics_csv), styles))
story.append(Spacer(1, 0.25*cm))
story.append(Paragraph(
    "Interpretation note: the custom landmark plausibility metric is usable for relative comparison here, while modified-run FID/KID should be described carefully if they appear numerically unstable.",
    styles["BodySmall"]
))
story.append(PageBreak())

# Charts page
story.append(Paragraph("Metric Charts", styles["HeadingSmall"]))
story.append(rl_image(fid_chart, max_width=17*cm, max_height=7*cm))
story.append(source_paragraph(str(metrics_csv), styles))
story.append(Spacer(1, 0.2*cm))
story.append(rl_image(lpips_chart, max_width=17*cm, max_height=7*cm))
story.append(source_paragraph(str(metrics_csv), styles))
story.append(Spacer(1, 0.2*cm))
story.append(rl_image(lm_chart, max_width=17*cm, max_height=7*cm))
story.append(source_paragraph(str(metrics_csv), styles))
story.append(PageBreak())

# Model sample pages
for title, fig_path, src in [
    ("Baseline Sample Grid", baseline_grid, str(baseline_dir)),
    ("Landmark-Regularized Sample Grid", landmark_grid, str(landmark_dir)),
    ("Frequency Ablation Sample Grid", freq_grid, str(freq_dir)),
    ("Qualitative Comparison Grid", comparison_grid, f"{baseline_dir} | {landmark_dir} | {freq_dir}"),
]:
    story.append(Paragraph(title, styles["HeadingSmall"]))
    story.append(rl_image(fig_path, max_width=18*cm, max_height=22*cm))
    story.append(Spacer(1, 0.2*cm))
    story.append(source_paragraph(src, styles))
    story.append(PageBreak())

# Personal page
story.append(Paragraph("Personal Image / Personal-Face Section", styles["HeadingSmall"]))
story.append(rl_image(personal_fig, max_width=18*cm, max_height=20*cm))
story.append(Spacer(1, 0.2*cm))
story.append(source_paragraph(personal_source, styles))
if personal_attack_summary.exists():
    story.append(Spacer(1, 0.25*cm))
    story.append(Paragraph(
        "This page includes the personal-face experiment outputs detected in the project. The CSV also stores attack-success-style and visual-similarity statistics when available.",
        styles["BodySmall"]
    ))
story.append(PageBreak())

# File manifest page
story.append(Paragraph("Artifact Manifest", styles["HeadingSmall"]))
manifest_lines = [
    f"CSV output: {output_csv}",
    f"PDF output: {output_pdf}",
    f"Metrics CSV source: {metrics_csv}",
    f"Metrics JSON source: {metrics_json if metrics_json.exists() else 'not found'}",
    f"Baseline image source directory: {baseline_dir}",
    f"Landmark-reg image source directory: {landmark_dir}",
    f"Freq-ablation image source directory: {freq_dir}",
    f"Personal raw image: {personal_raw if personal_raw.exists() else 'not found'}",
]
if personal_attack_pairs.exists():
    manifest_lines.append(f"Personal attack CSV source: {personal_attack_pairs}")
if personal_attack_summary.exists():
    manifest_lines.append(f"Personal attack summary JSON source: {personal_attack_summary}")

for line in manifest_lines:
    story.append(Paragraph(line.replace("&", "&amp;"), styles["BodySmall"]))
    story.append(Spacer(1, 0.12*cm))

# Footer helper
def add_page_number(canvas, doc):
    canvas.saveState()
    canvas.setFont("Helvetica", 9)
    canvas.drawRightString(A4[0] - 1.5*cm, 1.0*cm, f"Page {doc.page}")
    canvas.restoreState()

doc.build(story, onFirstPage=add_page_number, onLaterPages=add_page_number)

print("\nCreated files:")
print("CSV :", output_csv)
print("PDF :", output_pdf)
print("\nDone.")

FileNotFoundError: [Errno 2] No such file or directory: 'all_experiment_results.csv'